In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType
from datetime import datetime
from pyspark.sql.functions import from_json, col, get_json_object, current_timestamp


In [ ]:
event_schema = StructType([
        StructField("timestamp", StringType(), True),
        StructField("visitorid", StringType(), True),
        StructField("event", StringType(), True),
        StructField("itemid", StringType(), True),
        StructField("transactionid", StringType(), True),
        StructField("_ingestion_id", StringType(), True),
        StructField("_ingestion_timestamp", StringType(), True)
    ])

In [ ]:
event_schema = StructType([
        StructField("timestamp", StringType(), True),
        StructField("visitorid", StringType(), True),
        StructField("event", StringType(), True),
        StructField("itemid", StringType(), True),
        StructField("transactionid", StringType(), True),
        StructField("_ingestion_id", StringType(), True),
        StructField("_ingestion_timestamp", StringType(), True)
    ])

In [ ]:
df_parsed = df_kinesis.select(
    from_json(col("data").cast("string"), event_schema).alias("event_data"),
    col("kinesis.approximateArrivalTimestamp").alias("kinesis_timestamp")
).select(
    col("event_data.*"), 
    col("kinesis_timestamp")
)

df_enriched = df_parsed.withColumn("_processing_time", current_timestamp())

In [ ]:

query = df_enriched.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_LOCATION) \
    .option("mergeSchema", "true") \
    .table(TARGET_TABLE)

